# Rejected experiment workflow

This notebook tests a generic volatility filter and rejects it when validation does not improve.
It avoids private thresholds, exact research figures, and production settings.

In [ ]:
from pathlib import Path

import pandas as pd

from confscan.backtest.engine import run_backtest
from confscan.backtest.metrics import Metrics
from confscan.signals.ta import atr, detect_cross, ema

df = pd.read_csv(Path("tests/fixtures/btc_4h_sample.csv"), parse_dates=["timestamp"]).set_index(
    "timestamp"
)
cross = detect_cross(ema(df["close"], 12), ema(df["close"], 26))
base = run_backtest(df, cross == 1, cross == -1)

volatility = atr(df).fillna(method="bfill") / df["close"]
calm_market = volatility <= volatility.expanding().median()
filtered = run_backtest(df, (cross == 1) & calm_market, cross == -1)

pd.DataFrame(
    [
        {
            "idea": "baseline generic cross",
            "sharpe": Metrics.from_result(base).sharpe,
            "trades": len(base.trades),
        },
        {
            "idea": "generic calm-market filter",
            "sharpe": Metrics.from_result(filtered).sharpe,
            "trades": len(filtered.trades),
        },
    ]
)

A filter that reduces trades without improving validation quality should be rejected.
The output is deliberately generic; the reusable habit is the value.